In [20]:
##                      NAMED ENTITY RECOGNITION USING GLINER
#GLiNER is a Named Entity Recognition (NER) model capable of identifying any entity type using 
#a bidirectional transformer encoder (BERT-like). It provides a practical alternative to traditional 
#NER models, which are limited to predefined entities, and Large Language Models (LLMs) that, despite 
#their flexibility, are costly and large for resource-constrained scenarios.

In [ ]:
!pip install gliner==0.1.12

In [ ]:
import json
import os
import torch
from tqdm import tqdm
from gliner import GLiNER
from types import SimpleNamespace
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from torch.nn import CrossEntropyLoss
import unittest

with open("/kaggle/input/train-platinum-json/train_platinum.json", "r") as f:
    articles = json.load(f)

#tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

def preprocess_data(articles):
    processed_data = []
    for article_id, article_info in articles.items():
        title_text = article_info["metadata"]["title"]
        abstract_text = article_info["metadata"]["abstract"]
        
        combined_text = title_text + " " + abstract_text
        
        encoding = tokenizer(combined_text, return_offsets_mapping=True, add_special_tokens=False)
        tokenized_text = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
        offset_mapping = encoding["offset_mapping"]

        new_ner = []
        for ent in article_info["entities"]:
            if ent["location"] not in ["title", "abstract"]:
                continue

            start_char = ent["start_idx"]
            end_char = ent["end_idx"]
            label = ent["label"].lower()
            if ent["location"] == "abstract":
                offset_adjust = len(title_text) + 1
                start_char += offset_adjust
                end_char += offset_adjust
                
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                if token_end > start_char and token_start < end_char:
                    new_ner.append((idx, idx, label))
        
        processed_data.append({
            "tokenized_text": tokenized_text,
            "ner": new_ner,
        })
    return processed_data

processed_data = preprocess_data(articles)
model = GLiNER.from_pretrained("numind/NuNerZero")

# --------------------------------------------------
# 4. Split data into training, validation, and test sets
# --------------------------------------------------
train_data, val_data = train_test_split(processed_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(val_data, test_size=0.5, random_state=42)

# --------------------------------------------------
# 5. Define hyperparameters and training configuration
# --------------------------------------------------
config = SimpleNamespace(
    num_steps=1500,   
    eval_every=100,    
    train_batch_size=2,  
    max_len=600,          
    device='cuda',
    warmup_ratio=0.1,
    lr_encoder=1e-5,
    lr_others=5e-5,
    freeze_token_rep=True,
    max_types=25,
    shuffle_types=True,
    random_drop=True,
    max_neg_type_ratio=1,
)

# --------------------------------------------------
# 6. Training function (without saving logs/checkpoints during training)
# --------------------------------------------------
def train(model, config, train_data, eval_data=None):
    model = model.to(config.device)

    model.set_sampling_params(
        max_types=config.max_types,
        shuffle_types=config.shuffle_types,
        random_drop=config.random_drop,
        max_neg_type_ratio=config.max_neg_type_ratio,
        max_len=config.max_len
    )

    model.train()
    train_loader = model.create_dataloader(train_data, batch_size=config.train_batch_size, shuffle=True)
    optimizer = model.get_optimizer(config.lr_encoder, config.lr_others, config.freeze_token_rep)
    pbar = tqdm(range(config.num_steps))
    num_warmup_steps = int(config.num_steps * config.warmup_ratio) if config.warmup_ratio < 1 else int(config.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=config.num_steps
    )

    iter_train_loader = iter(train_loader)
    for step in pbar:
        try:
            batch = next(iter_train_loader)
        except StopIteration:
            iter_train_loader = iter(train_loader)
            batch = next(iter_train_loader)

        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(config.device)

        loss = model(batch)
        if torch.isnan(loss):
            continue

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        description = f"step: {step} | epoch: {step // len(train_loader)} | loss: {loss.item():.2f}"
        pbar.set_description(description)

        if (step + 1) % config.eval_every == 0 and eval_data is not None:
            model.eval()
            results, f1 = model.evaluate(
                eval_data["samples"],
                flat_ner=True,
                threshold=0.5,
                batch_size=12,
                entity_types=eval_data["entity_types"]
            )
            print(f"Step={step}\n{results}")
            model.train()

# --------------------------------------------------
# 7. Setup evaluation data
# --------------------------------------------------
unique_labels = list({label for d in processed_data for (_, _, label) in d["ner"]})
eval_data = {
    "entity_types": unique_labels,
    "samples": processed_data[:10]
}

# --------------------------------------------------
# 8. Train the model and save the final trained model
# --------------------------------------------------
train(model, config, processed_data, eval_data)
model.save_pretrained("NuZero_plat_v1")


In [ ]:
#testttttttttttttttttttttttt 1
import os
from gliner import GLiNER
from transformers import AutoTokenizer

# Change current working directory to the parent folder of your model
model_path = "/kaggle/input/nuzero_tfv2/pytorch/default/1/NuZero_tfv1"

# Now load the model using its folder name (which is now valid)
model = GLiNER.from_pretrained(model_path)
model.eval()  # Set model to evaluation/inference mode

# Load the tokenizer (make sure it matches the one used during training)
#tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large", local_files_only=True)

# Now you can use your model for inference
text = (
    "Repeated Social Defeat Stress Induces an Inflammatory Gut Milieu by Altering the Mucosal Barrier Integrity and Gut Microbiota Homeostasis.Posttraumatic stress disorder (PTSD) is a mental health condition triggered by exposure to traumatic events in an individual's life. Patients with PTSD are also at a higher risk for comorbidities. However, it is not well understood how PTSD affects human health and/or promotes the risk for comorbidities. Nevertheless, patients with PTSD harbor a proinflammatory milieu and dysbiotic gut microbiota. Gut barrier integrity helps to maintain normal gut homeostasis and its dysregulation promotes gut dysbiosis and inflammation. We used a mouse model of repeated social defeat stress (RSDS), a preclinical model of PTSD. Behavioral studies, metagenomics analysis of the microbiome, gut permeability assay (on mouse colon, using an Ussing chamber), immunoblotting, and immunohistochemical analyses were performed. Polarized intestinal epithelial cells and 3-dimensional crypt cultures were used for mechanistic analysis. The RSDS mice harbor a heightened proinflammatory gut environment and microbiota dysbiosis. The RSDS mice further showed significant dysregulation of gut barrier functions, including transepithelial electrical resistance, mucin homeostasis, and antimicrobial responses. RSDS mice also showed a specific increase in intestinal expression of claudin-2, a tight junction protein, and epinephrine, a stress-induced neurotransmitter. Treating intestinal epithelial cells or 3-dimensional cultured crypts with norepinephrine or intestinal luminal contents (fecal contents) upregulated claudin-2 expression and inhibited transepithelial electrical resistance. Traumatic stress induces dysregulation of gut barrier functions, which may underlie the observed gut microbiota changes and proinflammatory gut milieu, all of which may have an interdependent effect on the health and increased risk of comorbidities in patients with PTSD."
)

# If GLiNER provides a predict_entities method, you can do:
labels = [
    "biomedical technique", "microbiome", "DDF", "anatomical location",
    "dietary supplement", "food", "chemical", "bacteria", "human"
]
predicted_entities = model.predict_entities(text, labels)

print("Predicted Entities:")
for entity in predicted_entities:
    print(entity["text"], "=>", entity["label"])


In [ ]:
#TESTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT 2
from gliner import GLiNER

def merge_entities(entities):
    if not entities:
        return []
    merged = []
    current = entities[0]
    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (next_entity['start'] == current['end'] + 1 or next_entity['start'] == current['end']):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity
    # Append the last entity
    merged.append(current)
    return merged

model_path = "/kaggle/input/nuzero_for_your_usecase/pytorch/default/1/NuZero_for_your_usecase"
model = GLiNER.from_pretrained(model_path)

# NuZero requires labels to be lower-cased!
labels = ["biomedical technique", "microbiome", "DDF", "anatomical location",
    "dietary supplement", "food", "chemical", "bacteria", "human"]
labels = [l.lower() for l in labels]

text = "Repeated Social Defeat Stress Induces an Inflammatory Gut Milieu by Altering the Mucosal Barrier Integrity and Gut Microbiota Homeostasis.Posttraumatic stress disorder (PTSD) is a mental health condition triggered by exposure to traumatic events in an individual's life. Patients with PTSD are also at a higher risk for comorbidities. However, it is not well understood how PTSD affects human health and/or promotes the risk for comorbidities. Nevertheless, patients with PTSD harbor a proinflammatory milieu and dysbiotic gut microbiota. Gut barrier integrity helps to maintain normal gut homeostasis and its dysregulation promotes gut dysbiosis and inflammation. We used a mouse model of repeated social defeat stress (RSDS), a preclinical model of PTSD. Behavioral studies, metagenomics analysis of the microbiome, gut permeability assay (on mouse colon, using an Ussing chamber), immunoblotting, and immunohistochemical analyses were performed. Polarized intestinal epithelial cells and 3-dimensional crypt cultures were used for mechanistic analysis. The RSDS mice harbor a heightened proinflammatory gut environment and microbiota dysbiosis. The RSDS mice further showed significant dysregulation of gut barrier functions, including transepithelial electrical resistance, mucin homeostasis, and antimicrobial responses. RSDS mice also showed a specific increase in intestinal expression of claudin-2, a tight junction protein, and epinephrine, a stress-induced neurotransmitter. Treating intestinal epithelial cells or 3-dimensional cultured crypts with norepinephrine or intestinal luminal contents (fecal contents) upregulated claudin-2 expression and inhibited transepithelial electrical resistance. Traumatic stress induces dysregulation of gut barrier functions, which may underlie the observed gut microbiota changes and proinflammatory gut milieu, all of which may have an interdependent effect on the health and increased risk of comorbidities in patients with PTSD."

entities = model.predict_entities(text, labels)

entities = merge_entities(entities)

for entity in entities:
    print(entity["text"], "=>", entity["label"])


In [ ]:
import json
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

# Configuration
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5

# 1. Data Preparation
class RelationDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        self.relations = self._load_relations()
        self.label2id = {label: idx for idx, label in enumerate(self.get_all_relations())}
        
    def get_all_relations(self):
        return list(set([rel["predicate"] for article in self.data.values() for rel in article["relations"]])) + ["no_relation"]

    def _load_relations(self):
        processed = []
        for article_id, article in self.data.items():
            text = f"Title: {article['metadata']['title']} Abstract: {article['metadata']['abstract']}"
            entities = article["entities"]
            
            # Create entity map for quick lookup
            entity_map = {(e['start_idx'], e['end_idx']): e for e in entities}
            
            for rel in article["relations"]:
                # Get subject and object entities
                subj = entity_map[(rel['subject_start_idx'], rel['subject_end_idx'])]
                obj = entity_map[(rel['object_start_idx'], rel['object_end_idx'])]
                
                # Add special markers around entities
                marked_text = self._insert_markers(text, subj, obj)
                
                processed.append({
                    "text": marked_text,
                    "label": rel["predicate"],
                    "subject": subj["text_span"],
                    "object": obj["text_span"]
                })
        return processed

    def _insert_markers(self, text, subj, obj):
        # Insert entity markers around the subject and object
        marked_text = (
            text[:subj['start_idx']] + "[E1]" +
            text[subj['start_idx']:subj['end_idx']] + "[/E1]" +
            text[subj['end_idx']:obj['start_idx']] + "[E2]" +
            text[obj['start_idx']:obj['end_idx']] + "[/E2]" +
            text[obj['end_idx']:]
        )
        return marked_text

    def __len__(self):
        return len(self.relations)

    def __getitem__(self, idx):
        item = self.relations[idx]
        encoding = self.tokenizer(
            item["text"],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.label2id[item["label"]], dtype=torch.long)
        }

# 3. Training Loop
def train(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
    return total_loss / len(dataloader)

# 4. Evaluation
def evaluate(model, dataloader, device):
    model.eval()
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            
    return f1_score(true_labels, predictions, average="weighted")

# 5. Main Execution
if __name__ == "__main__":
    # Load your dataset
    with open("/kaggle/input/train-bronze/train_bronze.json") as f:
        data = json.load(f)
    
    # Initialize tokenizer
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    
    # Create dataset and dataloader
    dataset = RelationDataset(data, tokenizer)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # Setup model with the correct number of labels
    num_labels = len(dataset.label2id)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(EPOCHS):
        loss = train(model, dataloader, optimizer, device)
        f1 = evaluate(model, dataloader, device)
        print(f"Epoch {epoch+1}: Loss: {loss:.4f}, F1: {f1:.4f}")
    
    # Save model and tokenizer
    model.save_pretrained("bronze_RE")
    tokenizer.save_pretrained("bronze_RE")
